# 01 · Colab 真实层初步验证

**已提供：** 使用真实 HF LLaMA 权重和 WikiText-2 train 激活，提取完整通道的局部面板，测量 q/up 投影的 Identity/Hadamard RTN 误差与尺度稳定组比例。down projection 仅采集，不错误地把 FFN 输入旋转冒充全局残差 R1。

**待 Codex 实现：** 全局共享 R1 候选优化、输出侧 `Q B`、匹配参数与计算预算的 STE/grid 对照、held-out 评估、GPTQ 迁移及 G1 汇总。当前 probe 只是诊断，不能自动获得 G1 PASS。

默认 `RUN_REAL_MODEL=False`，整本执行只记录 SKIPPED，不下载 7B 权重。T4/L4/A100 的可用性与内存以实际报告为准；内存不足不允许偷偷换成已有量化基础模型。

## 路径与环境配置

CPU 默认路径不下载模型、不调用付费服务、不自动升级 PyTorch。**只上传 ipynb 不够，必须一起提供 src/configs/docs。** 首次在 Colab 上传整个 ZIP（或放 Drive），设置 `KIT_ARCHIVE`，将 `EXTRACT_ARCHIVE=True`，即可用下面单元安全解压；Drive 路径先设置 `MOUNT_DRIVE=True`。如果已经手动解压，则直接设置 `PROJECT_ROOT`。缺少基础 Python 依赖时才把 `INSTALL_BASE_DEPENDENCIES=True`。Colab 断线会丢失 `/content`，需要在阶段完成后同步结果；高频权重与激活 I/O 应留在本地磁盘。

访问受限 LLaMA 权重需要你已有的 Hugging Face 访问权。凭据只通过环境变量或 Colab Secrets 提供，不能写入 notebook、配置、提示词、日志或 Git。

In [1]:
from pathlib import Path
import os, sys, json

# Colab: 解压整个项目到 /content/cellquant_research_kit；不要仅上传 ipynb。
# AutoDL: 建议 /root/autodl-tmp/cellquant_research_kit。
# 留 None 时依次查找当前目录、父目录和上述两个位置。
PROJECT_ROOT = os.environ.get("CELLQUANT_PROJECT_ROOT") or None
RUN_ROOT = os.environ.get("CELLQUANT_RUN_ROOT") or None
KIT_ARCHIVE = None  # 例：/content/drive/MyDrive/cellquant_research_kit.zip
EXTRACT_ARCHIVE = False  # 首次运行且 ZIP 已上传时才改 True
ARCHIVE_DESTINATION = "/content/cellquant_workspace" if Path("/content").exists() else str(Path.cwd().parent / "cellquant_workspace")
MOUNT_DRIVE = False
DURABLE_ROOT = None  # 例：/content/drive/MyDrive/cellquant_runs；不存权重或 token
INSTALL_BASE_DEPENDENCIES = False
SEED = 20261115

In [2]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive mounting skipped; outputs remain under RUN_ROOT until copied.")

Drive mounting skipped; outputs remain under RUN_ROOT until copied.


In [3]:
archive_roots = []
if EXTRACT_ARCHIVE:
    import zipfile, hashlib
    if not KIT_ARCHIVE or not Path(KIT_ARCHIVE).is_file():
        raise FileNotFoundError("先上传完整 ZIP 并设置 KIT_ARCHIVE")
    archive_path = Path(KIT_ARCHIVE)
    destination = Path(ARCHIVE_DESTINATION).resolve()
    digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
    receipt = destination / "archive_source_sha256.txt"
    if destination.exists() and any(destination.iterdir()):
        if not receipt.is_file() or receipt.read_text().strip() != digest:
            raise FileExistsError("目标目录已有不同内容；请设置新的 ARCHIVE_DESTINATION，避免覆盖修改")
        print("Same archive already extracted; preserving existing project edits.")
    else:
        with zipfile.ZipFile(archive_path) as archive:
            infos = archive.infolist()
            if sum(info.file_size for info in infos) > 2 * 2**30:
                raise ValueError("项目包解压体积超过 2GiB；请勿把基础权重放进代码包")
            for info in infos:
                target = (destination / info.filename).resolve()
                if not target.is_relative_to(destination) or (info.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError("Unsafe archive member")
            destination.mkdir(parents=True, exist_ok=True)
            archive.extractall(destination)
        receipt.write_text(digest + "\n")
    archive_roots = [destination, destination / "cellquant_research_kit"]

candidates = ([Path(PROJECT_ROOT)] if PROJECT_ROOT else []) + archive_roots + [
    Path.cwd(), Path.cwd().parent,
    Path("/content/cellquant_research_kit"),
    Path("/root/autodl-tmp/cellquant_research_kit"),
]
roots = [p.resolve() for p in candidates
         if (p / "pyproject.toml").is_file() and (p / "src/cellquant").is_dir()]
if not roots:
    raise FileNotFoundError("请先解压完整项目包，并设置 PROJECT_ROOT；只上传 notebook 不足以运行。")
PROJECT_ROOT = roots[0]
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.environ["CELLQUANT_PROJECT_ROOT"] = str(PROJECT_ROOT)

if INSTALL_BASE_DEPENDENCIES:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r",
                           str(PROJECT_ROOT / "requirements/base.txt")])

from cellquant.runtime import (atomic_json, environment_report, resolve_paths,
                              build_manifest, verify_manifest, copy_completed_run)
PATHS = resolve_paths(PROJECT_ROOT, RUN_ROOT, DURABLE_ROOT)
print("Project:", PATHS["project"])
print("Runs:", PATHS["runs"])

Project: /workspace/scratch/506a2af5b817/cellquant_research_kit
Runs: /workspace/scratch/506a2af5b817/cellquant_research_kit/runs


## 1. 选择真实诊断配置

依赖安装是显式操作，不自动替换系统 torch：可先在终端执行 `python -m pip install -r requirements/hf_optional.txt`。运行前在 Colab Secrets 放置 `HF_TOKEN`（或环境变量），并确认已获模型访问权。

默认只抓取 layer 0、短序列与少量权重行，属于 smoke。正式 G1 应按冻结协议扩展到早/中/晚层、独立验证文本与多个种子。可抽 token 和权重行，**不可抽掉组内通道再重算尺度**。

In [4]:
from dataclasses import fields
from cellquant.panels import ProbeConfig, run_real_layer_probe

RUN_REAL_MODEL = False  # 改 True 后才会真正加载授权模型和训练文本
config_path = PROJECT_ROOT / "configs/colab_pilot.json"
outer_config = json.loads(config_path.read_text()) if config_path.exists() else {}
probe_settings = outer_config.get("real_layer_probe", {})
known = {f.name for f in fields(ProbeConfig)}
unknown = set(probe_settings) - known
if unknown:
    raise ValueError(f"Unknown real_layer_probe configuration fields: {sorted(unknown)}")
probe_config = ProbeConfig(**probe_settings)
OUT = PATHS["runs"] / "01_real_layer_probe"
OUT.mkdir(parents=True, exist_ok=True)
print(probe_config)

ProbeConfig(model_id='meta-llama/Llama-2-7b-hf', revision='main', dataset_revision='main', layers=[0], projections=['q_proj', 'up_proj', 'down_proj'], sequence_length=256, sequences=2, capture_tokens=32, weight_rows=8, group_size=-1, random_pairs=8, bits=4, seed=20261115, expected_model_gib=12.6, local_text_file=None, save_panels=True)


In [5]:
if RUN_REAL_MODEL:
    probe_result = run_real_layer_probe(probe_config, OUT)
else:
    probe_result = {
        "schema_version":"1.0", "status":"SKIPPED",
        "evidence_kind":"NO_MODEL_RUN", "G1_status":"INSUFFICIENT_EVIDENCE",
        "reason":"RUN_REAL_MODEL=False; no real-model metrics were produced", "panels":[]
    }
    atomic_json(OUT / "probe_result.json", probe_result)
print(json.dumps(probe_result, ensure_ascii=False, indent=2))

{
  "schema_version": "1.0",
  "status": "SKIPPED",
  "evidence_kind": "NO_MODEL_RUN",
  "G1_status": "INSUFFICIENT_EVIDENCE",
  "reason": "RUN_REAL_MODEL=False; no real-model metrics were produced",
  "panels": []
}


## 2. 查看真实测量结果

只显示本次 probe 产生的数值，不填论文结果或演示随机数。`down_proj` 的 `CAPTURED_ONLY` 不是失败，也不是主算法完成：其全局 R1 作用于输出隐藏维度，必须在完整后端中单独处理。

Identity/Hadamard 比较使用冻结局部浮点输入和对称 RTN，不等于完整 W4A4KV4 模型，也不是官方 QuaRot/SpinQuant 复现。

In [6]:
rows = []
for panel in probe_result.get("panels", []):
    if "identity" not in panel:
        print(panel.get("name"), panel.get("status"), panel.get("reason"))
        continue
    rows.append({
        "panel":panel["name"],
        "identity_output_nmse":panel["identity"]["output_nmse"],
        "hadamard_output_nmse":panel["hadamard"]["output_nmse"],
        "weight_unstable_fraction":panel["screening_weight"]["unstable_fraction"],
        "activation_unstable_fraction":panel["screening_activation"]["unstable_fraction"],
        "FP_identity_error":panel["full_precision_identity_relative_error"],
    })
if rows:
    import pandas as pd
    display(pd.DataFrame(rows))
    pd.DataFrame(rows).to_csv(OUT / "local_diagnostics.csv", index=False)
else:
    print("没有真实层结果；不要将空表作为成功实验。")

没有真实层结果；不要将空表作为成功实验。


## 3. 本册输出为何不能直接触发 AutoDL 全量？

还缺候选方法相对同预算 STE/grid 的 held-out 效应量、输出侧代价、跨层/种子覆盖和 GPTQ 迁移。请让 Codex 按项目提示词实现后端，再生成 `runs/gates/G1_evidence.json`，调用：

```bash
python -m cellquant.gates evaluate --stage G1 \
  --evidence runs/gates/G1_evidence.json --output runs/gates/G1.json
```

不能把本 probe 的 JSON 改名冒充该文件。G1 PASS 只允许进入 AutoDL 的受限 canary，不允许直接启动完整论文矩阵。

In [7]:
decision = {
    "status":"INSUFFICIENT_EVIDENCE", "gate_id":"G1", "probe_only":True,
    "missing":["global_R1_candidate", "matched_STE_and_grid", "heldout_effect_size",
               "output_side_cost", "required_model_layer_seed_coverage", "GPTQ_transfer"],
    "next_step":"Implement the specified backend, then run the G1 evaluator on real evidence"
}
atomic_json(OUT / "probe_decision.json", decision)
print(json.dumps(decision, ensure_ascii=False, indent=2))
if PATHS["durable"]:
    print(copy_completed_run(OUT, PATHS["durable"] / "01_real_layer_probe"))

{
  "status": "INSUFFICIENT_EVIDENCE",
  "gate_id": "G1",
  "probe_only": true,
  "missing": [
    "global_R1_candidate",
    "matched_STE_and_grid",
    "heldout_effect_size",
    "output_side_cost",
    "required_model_layer_seed_coverage",
    "GPTQ_transfer"
  ],
  "next_step": "Implement the specified backend, then run the G1 evaluator on real evidence"
}
